# 04 — Sentiment Analysis (NYC Tweets)

Applies Cardiff NLP's Twitter-RoBERTa-base sentiment classifier to the 20,000
tweet sample. Each tweet is classified as positive, neutral, or negative with
a confidence score.

**Input:** `data/processed/nyc_sample_20k.csv`  
**Output:** `data/processed/nyc_sample_with_sentiment.csv`

In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm import tqdm
import torch


/Users/saanvikakde/FURI_SP26/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Tweet Sample

Load the 20,000 tweet sample produced in notebook 03.

In [2]:
nyc_sample = pd.read_csv("../data/processed/nyc_sample_20k.csv")

print("Tweets loaded:", len(nyc_sample))
nyc_sample.head()


Tweets loaded: 20000


,user_id,tweet_id,text,timestamp
0,23532537,4359057774,Mansion tonite tweeple! DJ Mr Mauricio and a v...,2009-09-24 21:52:48
1,31056691,9466378487,RT @HEARTbreakPAM: Watching The Ringer & getti...,2010-02-22 01:12:09
2,48388361,6614507963,This is @andrearmani hackin ejaysz twitter!,2009-12-12 18:52:00
3,24788891,3988076198,@mrszBreunaXo Wuz good?,2009-09-14 15:39:30
4,58725560,3808152406,@thaG5 pin:30BC66CE,2009-09-06 19:15:23


## Step 2: Load Sentiment Model

Uses Cardiff NLP's `twitter-roberta-base-sentiment-latest` model via Hugging
Face Transformers. This model is fine-tuned specifically on tweets, making it
more appropriate than a general-purpose BERT model for this dataset.

Runs on GPU if available, otherwise falls back to CPU. The pooler weight
warnings are expected and do not affect classification output.

In [3]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 if torch.cuda.is_available() else -1
)


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


## Step 3: Clean Text

Strip leading/trailing whitespace. Non-string values (NaN) are replaced with
empty strings to prevent errors during inference.

In [4]:
def clean_text(text):
    if isinstance(text, str):
        return text.strip()
    return ""

nyc_sample["clean_text"] = nyc_sample["text"].apply(clean_text)


## Step 4: Run Inference in Batches

Processes tweets in batches of 32 to avoid memory errors. `tqdm` tracks
progress — full inference over 20,000 tweets takes approximately 12-15 minutes
on CPU.

In [5]:
def run_sentiment(texts, batch_size=32):
    results = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch_results = sentiment_model(batch)
        results.extend(batch_results)
    
    return results

In [6]:
texts = nyc_sample["clean_text"].tolist()

sentiment_results = run_sentiment(texts)


100%|█████████████████████████████████████████████████████████████████████████████████| 625/625 [08:35<00:00,  1.21it/s]


## Step 5: Extract Labels and Scores

Pull the predicted label (`positive`, `neutral`, `negative`) and confidence
score from each result and add them as new columns on the dataframe.

In [7]:
nyc_sample["sentiment_label"] = [r["label"] for r in sentiment_results]
nyc_sample["sentiment_score"] = [r["score"] for r in sentiment_results]

nyc_sample.head(5)


,user_id,tweet_id,text,timestamp,clean_text,sentiment_label,sentiment_score
0,23532537,4359057774,Mansion tonite tweeple! DJ Mr Mauricio and a v...,2009-09-24 21:52:48,Mansion tonite tweeple! DJ Mr Mauricio and a v...,positive,0.818303
1,31056691,9466378487,RT @HEARTbreakPAM: Watching The Ringer & getti...,2010-02-22 01:12:09,RT @HEARTbreakPAM: Watching The Ringer & getti...,negative,0.898204
2,48388361,6614507963,This is @andrearmani hackin ejaysz twitter!,2009-12-12 18:52:00,This is @andrearmani hackin ejaysz twitter!,neutral,0.738387
3,24788891,3988076198,@mrszBreunaXo Wuz good?,2009-09-14 15:39:30,@mrszBreunaXo Wuz good?,neutral,0.780780
4,58725560,3808152406,@thaG5 pin:30BC66CE,2009-09-06 19:15:23,@thaG5 pin:30BC66CE,neutral,0.832530


## Step 6: Map Labels to Numeric Values

Convert string labels to numeric scores for use in correlation analysis:
- `positive` → +1
- `neutral` → 0
- `negative` → −1

The mean numeric sentiment across all 20,000 tweets is **+0.064**, indicating
a slight positive bias in the dataset overall.

In [8]:
label_map = {
    "negative": -1,
    "neutral": 0,
    "positive": 1
}

nyc_sample["sentiment_numeric"] = nyc_sample["sentiment_label"].map(label_map)


In [9]:
nyc_sample["sentiment_numeric"].mean()


np.float64(0.06355)

## Step 7: Save Output

Save the full dataframe with sentiment labels, confidence scores, and numeric
values to the processed data directory for use in notebooks 05 and 07.

In [10]:
nyc_sample.to_csv(
    "../data/processed/nyc_sample_with_sentiment.csv",
    index=False
)